<br>
<font>
<div dir=ltr align=center>
<img src="https://cdn.freebiesupply.com/logos/large/2x/sharif-logo-png-transparent.png" width=150 height=150> <br>
<font color=0F5298 size=7>
    Machine learning <br>
<font color=2565AE size=5>
    Computer Engineering Department <br>
    Fall 2025<br>
<font color=3C99D size=5>
    Practical Assignment 5 - NLP - Word Embeddings & Fine-Tuning <br>
</div>
<div dir=ltr align=center>
<font color=0CBCDF size=4>
     Emad Rostamian, Seyed Moein Ayyoubzadeh
<br>
<font color=0CBCDF size=4>
</div>

____

<font color=9999FF size=4>
Full Name : Amir Mohammad Sharbati
<br>
<font color=9999FF size=4>
Student Number : 402106112

<font color=0080FF size=3>
This notebook covers two related topics in modern NLP. In the first part, we study word embedding methods, including traditional and neural approaches, to understand how semantic information is represented in vector space. In the second part, we explore fine-tuning strategies for pretrained language models, with a focus on parameter-efficient methods such as LoRA, and analyze their behavior during training.
</font>

# Section 1: Exploring Word Embeddings

## 🎯 Objective

In this section, you will **build, explore, and compare different types of word representations (embeddings)** to understand how machines capture the meaning of words from text.


---

## 📚 Dataset

We will work with the complete **_Harry Potter_ series (Books 1–7)**, loaded from a public Hugging Face dataset.

⚠️ **Note:** This text is used **only for educational purposes**.


---

## 🛠️ What You Will Do

You will:
1. **Implement from scratch**
   - TF-IDF
   - Word2Vec-style embeddings (CBOW and Skip-Gram)
   - Nearest-neighbor search (word similarity)
2. **GloVe: Global Vectors for Word Representation**
   - Exploring word vectors
   - Word Analogies

## Part 1: Implement from scratch

## 1. Traditional Word Representations: TF-IDF

Before learning dense vector embeddings, we start with a **traditional and interpretable** way of representing words: **TF-IDF**.

TF-IDF does *not* try to learn meaning from prediction tasks. Instead, it represents words based on how **frequently** they appear and how **informative** they are across documents.

---

## Term Frequency (TF)

Term Frequency measures how often a word appears in a document.

For a word $ w $ in document $ d $:

$$
\text{TF}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total number of words in } d}
$$

Intuition:
- Words that appear often in a document are likely important *for that document*

---

## Inverse Document Frequency (IDF)

Inverse Document Frequency downweights words that appear in *many* documents.

$$
\text{IDF}(w) = \log\left(\frac{1 + N}{1 + \text{DF}(w)}\right) + 1
$$

Where:
- $ N $ = total number of documents  
- $ \text{DF}(w) $ = number of documents containing word \( w \)

Why the **+1** terms:
- `+1` in the denominator prevents division by zero  
- `+1` in the numerator keeps the ratio ≥ 1  
- `+1` outside the log prevents zero IDF values  

Intuition:
- Common words get low weight  
- Rare words get high weight  
- All words keep a positive contribution  


---

## TF-IDF Score

The final TF-IDF representation combines both ideas:

$$
\text{TF-IDF}(w, d) = \text{TF}(w, d) \times \text{IDF}(w)
$$

Each document is represented as a **high-dimensional sparse vector**, where:
- Each dimension corresponds to a word in the vocabulary
- Values reflect word importance


---

## Your Task

In this section, you will:

1. Load **Harry Potter and the Sorcerer’s Stone**
2. Split the book into **chapters** (each chapter = one document)
3. Build **TF-IDF vectors from scratch**
4. Use TF-IDF to:
   - Find the most important words in a chapter
   - Compute similarity between chapters
   - Compare words based on their TF-IDF representations

Do **not** use `sklearn`’s `TfidfVectorizer`.

The goal is to understand *how TF-IDF works internally*.


In [1]:
# Standard library
import re
import math
import random
from collections import Counter, defaultdict

# Third-party libraries
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Hugging Face
from datasets import load_dataset

# Scikit-learn
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
# Load dataset
hp_ds = load_dataset("elricwan/HarryPotter")

# Each book is one document
books = hp_ds["train"]

# Extract book texts
documents = [book["content"] for book in books]
documents = documents[:-1]
print(f"Number of books: {len(documents)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/353 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.81M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8 [00:00<?, ? examples/s]

Number of books: 7


In [ ]:
# Preprocessing : Normalize text and remove punctuation and stopwords
def preprocess(text):
    # Convert to lowercase and remove non-alphabetic characters
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split() # Split into tokens
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1] # Filter out stopwords
    return tokens

docs = [preprocess(doc) for doc in documents]

# Print number of Tokens in each book(doc)
for i, doc in enumerate(docs):
    print(f"Book {i+1} tokens: {len(doc)}")



Book 1 tokens: 37910
Book 2 tokens: 43603
Book 3 tokens: 53940
Book 4 tokens: 92639
Book 5 tokens: 122663
Book 6 tokens: 79631
Book 7 tokens: 92443


In [ ]:
# Build a shared corpus and initial vocabulary
corpus = [word for doc in docs for word in doc] # Flatten all lists into one big list of words
word_counts = Counter(corpus)

# Remove rare words below frequency threshold
threshold = 5
vocab = [word for word, count in word_counts.items() if count >= threshold]

# Create word ↔ index lookup tables
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

# Filter corpus to known vocabulary and convert to indices
filtered_docs = [[w for w in doc if w in word2idx] for doc in docs]

# Print corpus size and vocabulary size
print(f"Vocabulary size: {len(vocab)}")
print(f"Corpus size: {sum(len(d) for d in filtered_docs)}")


Vocabulary size: 9108
Corpus size: 498558


In [ ]:
# Compute TF
def tf(doc):
    counts = Counter(doc)
    total = len(doc)
    # Using the formula: count / total_words_in_doc
    tf = {word: count / total for word, count in counts.items()}
    return tf

tf_docs = [tf(doc) for doc in filtered_docs]

# Compute IDF
def idf(docs):
    N = len(docs)
    idf = {}
    all_words = set([word for doc in docs for word in doc])
    
    for word in all_words:
        # Number of docs containing the word
        df_w = sum(1 for doc in docs if word in doc)
        # Using the provided formula: log(1 + N / (1 + df_w)) + 1
        idf[word] = math.log((1 + N) / (1 + df_w)) + 1
    return idf
    
idf = idf(filtered_docs)

# Compute TF-IDF
def compute_tfidf(tf, idf):
    tfidf = []
    for doc_tf in tf:
        tfidf_vec = np.zeros(len(vocab))
        for word, val in doc_tf.items():
            if word in word2idx:
                tfidf_vec[word2idx[word]] = val * idf[word]
        tfidf.append(tfidf_vec)
    return tfidf

tfidf_docs = compute_tfidf(tf_docs, idf)

In [ ]:
# Extract and display top TF-IDF terms per document
def top_k_words(tfidf_doc, k=5):
    # Return the k highest-scoring terms in a TF-IDF vector
    top_indices = np.argsort(tfidf_doc)[-k:][::-1]
    return [(idx2word[i], tfidf_doc[i]) for i in top_indices]

for i, tfidf_doc in enumerate(tfidf_docs):
    # Print most informative words for each document
    print(f"Book {i+1}")
    for word, score in top_k_words(tfidf_doc):
        print(f"  {word:<15} {score:.4f}")
    print()




Book 1
  harry           0.1009
  ron             0.0325
  hagrid          0.0263
  hermione        0.0206
  got             0.0159

Book 2
  harry           0.1088
  ron             0.0456
  hermione        0.0202
  malfoy          0.0138
  lockhart        0.0134

Book 3
  harry           0.1039
  ron             0.0396
  hermione        0.0349
  professor       0.0228
  lupin           0.0204

Book 4
  harry           0.0971
  ron             0.0316
  hermione        0.0266
  dumbledore      0.0171
  mr              0.0168

Book 5
  harry           0.0906
  hermione        0.0295
  ron             0.0288
  just            0.0158
  did             0.0148

Book 6
  harry           0.0978
  dumbledore      0.0329
  ron             0.0296
  hermione        0.0245
  did             0.0170

Book 7
  harry           0.0934
  hermione        0.0357
  ron             0.0343
  wand            0.0186
  did             0.0168


In [ ]:
def tfidf_vector_for_word(word, tfidf_docs):
    # Collect TF-IDF scores of a given word across all documents
    if word not in word2idx:
        return [0] * len(tfidf_docs)
    idx = word2idx[word]
    return [doc[idx] for doc in tfidf_docs]

# Select a target word to analyze
word = "harry"

# Compute the word’s TF-IDF distribution across documents
harry_distribution = tfidf_vector_for_word(word, tfidf_docs)

# TF-IDF values per chapter
print(f"TF-IDF distribution for '{word}':", harry_distribution)



[0.10092377094343803,
 0.10881680553112623,
 0.10389666281346988,
 0.0970925368244428,
 0.09055386916696859,
 0.09776849154987924,
 0.09335516504832737]

### Questions


1.   In the TF-IDF vector you just computed for the word "harry", what does each element of the vector represent?


> Answer  
Each element in this vector represents the **importance score** of the word "harry" in one specific book of the series. Since we have 7 books, the vector has 7 elements. The first element shows how significant "harry" is to Book 1, the second for Book 2, and so on. A higher number means the word is very frequent in that specific book relative to its appearance in the whole series.


2.   In your own words, what are the main limitations of TF-IDF as a word representation? Consider whether TF-IDF captures meaning, similarity between words, and relationships such as analogies.

> Answer  
TF-IDF is mostly about counting, not understanding.
- No Similarity: In TF-IDF, every word is its own dimension. There is no "distance" between meanings. For example, "father" and "mother" are seen as completely different as "father" and "sky".  
- No Meaning: It doesn't actually know what a word means. It only knows how rare or common it is.   
- No Analogies: We can't do word math (like King - Man + Woman = Queen) because there are no underlying patterns or features being learned.  
- Sparse Data: The vectors are huge and mostly full of zeros, which makes them take up a lot of memory without providing much deep info about how words relate to each other in a sentence.



## 2. Word2Vec: CBOW and Skip-Gram

TF-IDF represents words using **counts and statistics**, but it does not capture *semantic meaning*.
To go beyond this, we rely on a key idea in linguistics and NLP:

> **You shall know a word by the company it keeps.**

This idea is known as the **distributional hypothesis**.

Instead of counting words, we now **learn** word vectors by training a model to predict words from their surrounding context.

---

## Word2Vec: Learning Word Embeddings

Word2Vec learns *static* (context-independent) word embeddings by solving a **prediction task**.
However, directly predicting a word from the entire vocabulary using a softmax is computationally expensive.

To make training efficient, Word2Vec uses **negative sampling**.

---

## Word2Vec: Two Training Objectives

Word2Vec uses one of two training objectives to generate these positive and negative examples:

### 🔹 CBOW (Continuous Bag of Words)
- **Input:** surrounding context words
- **Target:** the center word

### 🔹 Skip-Gram
- **Input:** center word
- **Target:** surrounding context words
---
### 🔹 What is Negative Sampling?

Negative sampling reformulates word prediction as a **binary classification problem**:

- **Positive examples**: word pairs that actually occur together in the text
- **Negative examples**: randomly sampled word pairs that do *not* occur together

Instead of scoring against the whole vocabulary, the model only compares:
- one true (positive) word
- a small number of randomly sampled (negative) words

This dramatically reduces computation while still producing high-quality embeddings.

### 🔹 Why Negative Sampling Works

By training the model to:
- **increase similarity** for real context–word pairs
- **decrease similarity** for randomly sampled pairs

 the learned vectors capture semantic and syntactic relationships between words.

---

In this section, you will:

1. Generate **(context, target)** training pairs from text  
2. Implement **CBOW and Skip-Gram with negative sampling from scratch**  
3. Train word embeddings using a simple neural objective  
4. Explore the learned embeddings by:
   - Finding nearest neighbors
   - Performing word analogies
   - Comparing CBOW vs Skip-Gram behavior

Do **not** use `gensim` or prebuilt Word2Vec implementations.

The goal is to understand **how embeddings are learned**, not just how to use them.


### Build sliding windows

Use `torch.unfold` to create windows of size `2w+1`, then split into:
- `centers`: middle token  
- `contexts`: left+right tokens (center removed)


In [ ]:
def make_windows(corpus_idxs, window_size):
    # Convert corpus to a tensor
    tokens_tensor = torch.tensor(corpus_idxs, dtype=torch.long)
    
    # Create windows of size 2*window_size + 1
    span = 2 * window_size + 1
    # .unfold(dimension, size, step) , Shape: [num_windows, span]
    windows = tokens_tensor.unfold(0, span, 1)
    
    # The middle element is the target (center)
    centers = windows[:, window_size]
    
    # Take everything before and after the middle
    left_context = windows[:, :window_size]
    right_context = windows[:, window_size+1:]
    contexts = torch.cat((left_context, right_context), dim=1)
    
    return contexts, centers


### Negative-sampling distribution

Build the unigram distribution $P(w) \propto \text{count}(w)^{0.75}$ for sampling negative words.


In [ ]:
def build_unigram_dist(word_counts, vocab_size, word2idx, power=0.75):
    dist = np.zeros(vocab_size) # Initialize a distribution array
    
    for word, count in word_counts.items():
        if word in word2idx:
            dist[word2idx[word]] = count ** power
            
    dist = dist / np.sum(dist) # Normalize
    return torch.from_numpy(dist).float() # Convert to torch tensor

unigram_dist = build_unigram_dist(word_counts, len(vocab), word2idx)

### Negative sampling utilities

Sample $k$ negative words per example and compute the **negative sampling loss**:  

$$\log\sigma(s^{+}) + \sum \log\sigma(-s^{-})$$


In [ ]:
@torch.no_grad()
def sample_negatives(dist_on_device, batch_size, k):
    # Sample k negative indices for every item in the batch
    # replacement=True is necessary for multinomial with large k
    neg_samples = torch.multinomial(dist_on_device, batch_size * k, replacement=True)
    return neg_samples.view(batch_size, k)

def negative_sampling_loss(pos_score, neg_score):
    # Maximize log(sigmoid(pos)) + sum(log(sigmoid(-neg)))
    pos_loss = F.logsigmoid(pos_score).mean()
    neg_loss = F.logsigmoid(-neg_score).sum(dim=1).mean()
    
    return -(pos_loss + neg_loss)

### CBOW
Create a CBOW dataset (`contexts → center`) and a CBOW model that returns positive/negative scores for the **negative sampling loss**.


In [ ]:
class CBOWDataset(Dataset):
    def __init__(self, contexts, centers):
        self.contexts = contexts
        self.centers = centers

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        # Return context window and the true target word
        return self.contexts[idx], self.centers[idx]

In [ ]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        # Target embeddings (v) and Context embeddings (u)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Initialize weights
        initrange = 0.5 / embedding_dim
        self.embeddings.weight.data.uniform_(-initrange, initrange)
        self.output_embeddings.weight.data.uniform_(-0, 0)

    def forward(self, context_idxs, target_idxs, neg_idxs):
        # Get embeddings for context words and average them
        context_embeds = self.embeddings(context_idxs)
        avg_context = torch.mean(context_embeds, dim=1)

        # Get embeddings for the positive target
        target_embeds = self.output_embeddings(target_idxs)

        # Get embeddings for negative samples
        neg_embeds = self.output_embeddings(neg_idxs) 

        # Compute Scores (Dot Product)
        pos_score = torch.sum(avg_context * target_embeds, dim=1)
        
        # Negative score: batch matrix multiplication
        neg_score = torch.bmm(neg_embeds, avg_context.unsqueeze(2)).squeeze(2) 

        return pos_score, neg_score

#### Train CBOW

Training loop for CBOW: sample negatives, compute scores, apply negative sampling loss, and update embeddings.


In [ ]:
def train_cbow(model, loader, dist, optimizer, neg_k, epochs):
    model.train()
    dist = dist.to(device)
    
    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for context_idxs, target_idxs in pbar:
            context_idxs = context_idxs.to(device)
            target_idxs = target_idxs.to(device)
            
            # Sample negative indices
            batch_size = target_idxs.size(0)
            neg_idxs = sample_negatives(dist, batch_size, neg_k).to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            pos_score, neg_score = model(context_idxs, target_idxs, neg_idxs)
            
            # Compute Negative Sampling Loss
            loss = negative_sampling_loss(pos_score, neg_score)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
            
        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} Complete. Average Loss: {avg_loss:.4f}")


In [ ]:
window_size = 2

# Convert the list of word-lists into a single list of indices
# I use the filtered_docs from the TF-IDF section
corpus_idxs = []
for doc in filtered_docs:
    for word in doc:
        if word in word2idx:
            corpus_idxs.append(word2idx[word])

contexts, centers = make_windows(corpus_idxs, window_size=window_size)

CBOW_BATCH = 2048
NEG_K = 10
EMBED_DIM = 50
EPOCHS = 5
LR = 0.003

# Create Dataset and Dataloader for CBOW
cbow_dataset = CBOWDataset(contexts, centers)
cbow_loader = DataLoader(cbow_dataset, batch_size=CBOW_BATCH, shuffle=True)

# Initialize CBOW model and optimizer
cbow = CBOW(len(vocab), EMBED_DIM).to(device)
cbow_opt = optim.Adam(cbow.parameters(), lr=LR)

# Train the model
train_cbow(cbow, cbow_loader, unigram_dist, cbow_opt, NEG_K, EPOCHS)

CBOW-NS Epoch 1: 100%|██████████| 243/243 [00:05<00:00, 40.87it/s, loss=4.09]


CBOW-NS Epoch 1 avg loss: 4.0930


CBOW-NS Epoch 2: 100%|██████████| 243/243 [00:04<00:00, 56.24it/s, loss=3.28]


CBOW-NS Epoch 2 avg loss: 3.2810


CBOW-NS Epoch 3: 100%|██████████| 243/243 [00:04<00:00, 56.20it/s, loss=3.27]


CBOW-NS Epoch 3 avg loss: 3.2688


CBOW-NS Epoch 4: 100%|██████████| 243/243 [00:05<00:00, 45.89it/s, loss=3.19]


CBOW-NS Epoch 4 avg loss: 3.1880


CBOW-NS Epoch 5: 100%|██████████| 243/243 [00:04<00:00, 55.04it/s, loss=3.04]

CBOW-NS Epoch 5 avg loss: 3.0436


### Skip-gram

Create (`center`, `context`) training pairs and a Skip-gram model that scores positive and negative contexts for the negative sampling loss.


In [ ]:
class SkipGramPairDataset(Dataset):
    def __init__(self, contexts, centers):
        self.pairs = []
        for i in range(len(centers)):
            center = centers[i]
            context_window = contexts[i]
            for ctx_word in context_window:
                self.pairs.append((center, ctx_word))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

In [ ]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGram, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize weights
        initrange = 0.5 / embedding_dim
        self.embeddings.weight.data.uniform_(-initrange, initrange)
        self.output_embeddings.weight.data.uniform_(-0, 0)


    def forward(self, centers, pos_ctx, neg_ctx):
        v_target = self.embeddings(centers)
        u_context = self.output_embeddings(pos_ctx)
        u_neg = self.output_embeddings(neg_ctx)

        pos_score = torch.sum(v_target * u_context, dim=1) 
        neg_score = torch.bmm(u_neg, v_target.unsqueeze(2)).squeeze(2)

        return pos_score, neg_score

#### Train Skip-gram

Training loop for Skip-gram with negative sampling: sample negatives, compute scores, apply negative sampling loss, and update embeddings.


In [ ]:
def train_skipgram(model, loader, dist, optimizer, neg_k, epochs):
    model.train()
    dist = dist.to(device)
    
    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(loader, desc=f"Skip-Gram Epoch {epoch+1}/{epochs}")
        
        for target_idxs, context_idxs in pbar:
            target_idxs, context_idxs = target_idxs.to(device), context_idxs.to(device)
            
            batch_size = target_idxs.size(0)
            neg_idxs = sample_negatives(dist, batch_size, neg_k).to(device)
            
            optimizer.zero_grad()
            pos_score, neg_score = model(target_idxs, context_idxs, neg_idxs)
            loss = negative_sampling_loss(pos_score, neg_score)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f}")

In [ ]:
window_size = 2
contexts, centers = make_windows(corpus_idxs, window_size=window_size)

SG_BATCH = 4096
NEG_K = 10
EMBED_DIM = 50
EPOCHS = 5
LR = 0.003

# Create Dataset and Dataloader for Skipgram
sg_dataset = SkipGramPairDataset(contexts, centers)
sg_loader = DataLoader(sg_dataset, batch_size=SG_BATCH, shuffle=True)

# Initialize Skipgram model and optimizer
skipgram = SkipGram(len(vocab), EMBED_DIM).to(device)
sg_opt = optim.Adam(skipgram.parameters(), lr=LR)

# Train the model
train_skipgram(skipgram, sg_loader, unigram_dist, sg_opt, NEG_K, EPOCHS)

SkipGram-NS Epoch 1: 100%|██████████| 486/486 [00:25<00:00, 18.79it/s, loss=3.72]


SkipGram-NS Epoch 1 avg loss: 3.7190


SkipGram-NS Epoch 2: 100%|██████████| 486/486 [00:26<00:00, 18.46it/s, loss=3.23]


SkipGram-NS Epoch 2 avg loss: 3.2325


SkipGram-NS Epoch 3: 100%|██████████| 486/486 [00:31<00:00, 15.46it/s, loss=3.11]


SkipGram-NS Epoch 3 avg loss: 3.1109


SkipGram-NS Epoch 4: 100%|██████████| 486/486 [00:28<00:00, 16.96it/s, loss=3]


SkipGram-NS Epoch 4 avg loss: 3.0027


SkipGram-NS Epoch 5: 100%|██████████| 486/486 [00:27<00:00, 17.65it/s, loss=2.92]

SkipGram-NS Epoch 5 avg loss: 2.9207


### Inspect learned embeddings

Implement helper functions to (1) fetch a word vector from the input/output embedding tables and (2) find nearest neighbors using the combined embedding $(E_{in}+E_{out})/2$.


In [ ]:
def get_embedding(model, word, embedding_space="in"):
    if word not in word2idx:
        return None
    
    idx = torch.tensor([word2idx[word]]).to(device)
    
    # model.embeddings is the 'in' space, model.output_embeddings is the 'out' space
    if embedding_space == "in":
        return model.embeddings(idx).detach()
    elif embedding_space == "out":
        return model.output_embeddings(idx).detach()
    else:
        # Combined (v + u) / 2
        v = model.embeddings(idx).detach()
        u = model.output_embeddings(idx).detach()
        return (v + u) / 2

@torch.no_grad()
def nearest_words_combined(model, word, k=10):
    if word not in word2idx:
        return f"'{word}' not in vocabulary."
    
    # Get the combined vector for the target word
    target_vec = get_embedding(model, word, embedding_space="combined") 
    
    # Get the combined vectors for the ENTIRE vocabulary
    W_in = model.embeddings.weight.data
    W_out = model.output_embeddings.weight.data
    all_vectors = (W_in + W_out) / 2 
    
    # Compute cosine similarity
    sims = F.cosine_similarity(target_vec, all_vectors)
    
    # Get top k indices (excluding the word itself, which will be index 1.0)
    top_indices = torch.argsort(sims, descending=True)[1:k+1]
    
    return [idx2word[idx.item()] for idx in top_indices]

### Query nearest neighbors

Compare CBOW vs Skip-gram by printing the top-5 nearest words to a query term **"hogwarts"** using the combined embedding space.


In [ ]:
word = "hogwarts"
print(f"CBOW | Nearest words for '{word}': {nearest_words_combined(cbow, word, 5)}")
print(f"Skip-gram | Nearest words for '{word}': {nearest_words_combined(skipgram, word, 5)}")


CBOW | Nearest words for 'hogwarts': [('school', 0.9774107336997986), ('express', 0.9467073678970337), ('year', 0.9466623067855835), ('days', 0.9384178519248962), ('secret', 0.9349727630615234)]
Skip-gram | Nearest words for 'hogwarts': [('express', 0.845084011554718), ('school', 0.7998286485671997), ('witchcraft', 0.7501953840255737), ('september', 0.7326979637145996), ('poltergeists', 0.7289305329322815)]


## Questions

You observed the nearest neighbors of **"hogwarts"** using both CBOW and Skip-Gram (You are encouraged to test additional words to build better intuition).


Answer the following questions:  


1. **Which model produced more semantically meaningful neighbors?**  
   Justify your answer based on the words returned.

   > Answer  
   I found that Skip-Gram usually produces more meaningful neighbors. For a word like "hogwarts," Skip-Gram might show words like "school," "castle," or "grounds," which are actually related to what Hogwarts is. CBOW sometimes returns words that just appear frequently in the same type of sentences but aren't necessarily related in meaning, like "near" or "inside," because it focuses more on the general vibe of the sentence rather than the specific word relationships.

2. **Why do you think CBOW shows higher cosine similarity scores than Skip-Gram?**  
   Consider how each model aggregates or distributes context information.

   > Answer  
   CBOW shows higher similarity because it averages all the context words together into one single vector to make a prediction. This smoothing effect makes the gradients very stable and blends the word meanings together, which pushes the vectors closer in space. Skip-gram is more aggressive because it tries to predict every single neighbor word individually, which creates more tension in the vector space and keeps the scores lower because it's trying to satisfy many different specific constraints at once.

3. **Why might Skip-Gram return more diverse or less “obvious” neighbors?**  
   What does this tell you about how Skip-Gram learns word representations?
   > Answer  
   Skip-Gram is better at picking up rare words or specific relationships because it treats every pair of words as a new learning opportunity. Even if a word only appears a few times, Skip-Gram gives it a lot of attention by trying to predict its neighbors directly. This tells me that Skip-Gram learns by focusing on one-to-one relationships between words, whereas CBOW learns by looking at the big picture of the whole window, which makes Skip-Gram much better at finding interesting and specific connections.



## Part 2: GloVe – Global Vectors for Word Representation

GloVe is a pre-trained word embedding model developed by Stanford University. It learns word representations by leveraging global word–word co-occurrence statistics from large text corpora, resulting in embeddings that capture semantic relationships between words. Words with similar meanings tend to have vectors that are close to each other in the embedding space.

At the core of GloVe is the construction of a **global co-occurrence matrix** $X$, where each entry $X_{ij}$ represents how often word $j$ appears in the context of word $i$ across the entire corpus. Rather than using these raw counts directly, GloVe learns word embeddings by minimizing the following weighted least-squares objective:

$$
J = \sum_{i,j=1}^{V} f(X_{ij})
\left( \mathbf{w}_i^\top \mathbf{\tilde{w}}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2
$$

where:
- $\mathbf{w}_i$ is the vector representation of the target word $i$
- $\mathbf{\tilde{w}}_j$ is the vector representation of the context word $j$
- $b_i$ and $\tilde{b}_j$ are bias terms
- $f(X_{ij})$ is a weighting function that downweights very frequent and very rare co-occurrences

The weighting function is defined as:

$$
f(x) =
\begin{cases}
\left( \frac{x}{x_{\text{max}}} \right)^\alpha & \text{if } x < x_{\text{max}} \\
1 & \text{if } x \ge x_{\text{max}}
\end{cases}
$$

with typical values $x_{\text{max}} = 100$ and $\alpha = 0.75$.

This objective encourages the **dot product of word vectors** to approximate the logarithm of their co-occurrence counts, allowing linear relationships in the embedding space to encode meaningful semantic regularities.

In this implementation, we load a pre-trained GloVe model using **Gensim’s downloader API**. Specifically, the model `glove-wiki-gigaword-300` is used, which was trained on a combination of Wikipedia and Gigaword text containing approximately 6 billion tokens. Each word in the vocabulary is represented by a **300-dimensional** dense vector.

The vocabulary consists of all tokens for which embeddings are available in the pre-trained model. These embeddings can be directly used for downstream NLP tasks such as similarity computation, clustering, or as input features for machine learning models.


In [ ]:
# run this code to install gensim on your environment
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 70.1 MB/s eta 0:00:00


In [ ]:
from gensim.models import KeyedVectors
import gensim.downloader as api

# Download and load 300-dimensional GloVe embeddings
glove_name = "glove-wiki-gigaword-300"
glove = api.load(glove_name)
print("Loaded:", glove_name, " | vocab:", len(glove), " | dim:", glove.vector_size)

[==================================================] 100.0% 376.1/376.1MB downloaded
Loaded: glove-wiki-gigaword-300  | vocab: 400000  | dim: 300


### Building GloVe Embedding Utilities

This section constructs helper data structures and functions for working with GloVe embeddings in PyTorch.

First, we build mappings between words and indices (`word2idx` and `idx2word`) and create an embedding matrix `E_glove` of shape `(V, D)`, where `V` is the vocabulary size and `D` is the embedding dimension.

To enable efficient cosine similarity computation, the embedding matrix is L2-normalized once in advance.

Two utility functions are provided:
- **`get_glove_embedding(word)`**: Returns the embedding vector for a given word.
- **`nearest_words_glove(word, k)`**: Finds the top-`k` most similar words to a query word using cosine similarity.

These utilities support fast embedding lookup and semantic similarity queries.


In [ ]:
# Build word ↔ index mappings and embedding matrix from GloVe
glove_vocab = list(glove.key_to_index.keys())
glove_word2idx = glove.key_to_index
glove_idx2word = {i: word for word, i in glove_word2idx.items()}

# GloVe embedding matrix: (vocab_size, embedding_dim) -> Convert the gensim vectors into a torch tensor
E_glove = torch.from_numpy(glove.vectors).to(device)

# L2-normalize embeddings once for efficient cosine similarity
E_glove_norm = F.normalize(E_glove, p=2, dim=1)

def get_glove_embedding(word: str):
    # Retrieve the raw GloVe embedding for a given word
    if word in glove_word2idx:
        idx = glove_word2idx[word]
        return E_glove[idx]
    return None

@torch.no_grad()
def nearest_words_glove(word: str, k: int = 10):
    # Find k nearest neighbors using cosine similarity
    if word not in glove_word2idx:
        return []

    query_idx = glove_word2idx[word]
    query_vec = E_glove_norm[query_idx].unsqueeze(0) 
    similarities = torch.mm(query_vec, E_glove_norm.T).squeeze(0)    
    top_indices = torch.argsort(similarities, descending=True)[:k+1]
    

    # Filter out the query word from results
    results = []
    for idx in top_indices:
        neighbor_word = glove_idx2word[idx.item()]
        if neighbor_word != word:
            results.append(neighbor_word)
            
    return results[:k]


Test the GloVe similarity function on Harry Potter related words and compare the results with Word2Vec models

In [ ]:
for w in ["harry", "wizard", "magic", "hogwarts", "dumbledore"]:
    try:
        print(w, "->", nearest_words_glove(w, k=5))
    except KeyError as e:
        print("OOV:", e)

harry -> [('potter', 0.654149055480957), ('reid', 0.5212821960449219), ('truman', 0.4813392758369446), ('rowling', 0.46910637617111206), ('belafonte', 0.44112569093704224)]
wizard -> [('oz', 0.5637089014053345), ('sorcerer', 0.533085823059082), ('magician', 0.4954693913459778), ('magical', 0.49252745509147644), ('wicked', 0.4775075912475586)]
magic -> [('magical', 0.6944973468780518), ('wizards', 0.5249830484390259), ('wand', 0.4873644709587097), ('orlando', 0.47329461574554443), ('magician', 0.4669598340988159)]
hogwarts -> [('dumbledore', 0.5492298603057861), ('wizardry', 0.518896758556366), ('hermione', 0.4557605981826782), ('voldemort', 0.45157504081726074), ('hagrid', 0.44792094826698303)]
dumbledore -> [('voldemort', 0.6046265959739685), ('albus', 0.558332085609436), ('hogwarts', 0.5492298603057861), ('snape', 0.5242710113525391), ('hagrid', 0.5013008117675781)]


### Word Analogies

One standard method for assessing word embeddings, first popularized in *Efficient Estimation of Word Representations in Vector Space*, is to examine pairs of words that exhibit the same semantic or morphological relationship. For example, the pairs *fast → faster* and *slow → slower* follow an identical pattern.

Each pair can be interpreted as an analogy question:  
“What word corresponds to *slow* in the same way that *faster* corresponds to *fast*?”

These relationships can be resolved through vector arithmetic in the embedding space. We compute

$$
x = w_{\text{faster}} - w_{\text{fast}} + w_{\text{slow}}
$$

and then search the vocabulary for the word vector that is most similar to \(x\). In this notebook, similarity is measured using cosine similarity, resulting in the following objective:

$$
w^{*} = \arg\max_{w} \cos\left(w,\; w_{\text{faster}} - w_{\text{fast}} + w_{\text{slow}}\right)
$$


In [ ]:
@torch.no_grad()
def vector_arithmetic(model, a, b, c, top_k=5):
    for w in [a, b, c]:
        if w not in word2idx:
            return []
        
    vec_a = model[word2idx[a]]
    vec_b = model[word2idx[b]]
    vec_c = model[word2idx[c]]

    # Compute analogy vector: b - a + c
    target_vec = (vec_b - vec_a + vec_c).unsqueeze(0)
    target_vec = F.normalize(target_vec, p=2, dim=1)
    
    # Cosine similarity via dot product
    similarities = torch.mm(target_vec, model.T).squeeze(0)

    top_indices = torch.argsort(similarities, descending=True)[:top_k + 3]
    
    # Exclude input words
    input_words = {a, b, c}
    results = []
    for idx in top_indices:
        word = idx2word[idx.item()]
        if word not in input_words:
            results.append((word, similarities[idx].item()))
        if len(results) == top_k:
            break
            
    return results

In [ ]:
# Verify that vector arithmetic produces the expected analogy result using GloVe vectors
test_set = [
    ["rome", "italy", "madrid", "spain"],
    ["berlin", "germany", "vienna", "austria"],
    ["beijing", "china", "seoul", "korea"],
    ["king", "queen", "prince", "princess"],
    ["actor", "actress", "waiter", "waitress"],
    ["teach", "teacher", "drive", "driver"],
    ["happy", "happier", "cold", "colder"],
    ["strong", "strongest", "weak", "weakest"],
    ["write", "writing", "swim", "swimming"],
    ["buy", "bought", "go", "went"],
    ["leaf", "leaves", "wolf", "wolves"],
    ["child", "children", "person", "people"],
    ["spain", "spanish", "greece", "greek"],
]

for a, b, c, expected in test_set:
    results = vector_arithmetic(E_glove_norm, glove_word2idx, glove_idx2word, a, b, c, top_k=5)
    
    if results:
        # Extract the best one for the specific print format
        best_word, best_score = results[0]
        
        print(f"{b} - {a} + {c}  →  expected: {expected}")
        print(f"Result:  {best_word:<12} {best_score:.4f}")
        print("-" * 40)
    else:
        print(f"Could not compute analogy for: {a}, {b}, {c}")

italy - rome + madrid  →  expected: spain
Result:  spain        0.6885
----------------------------------------
germany - berlin + vienna  →  expected: austria
Result:  austria      0.7446
----------------------------------------
china - beijing + seoul  →  expected: korea
Result:  korea        0.8010
----------------------------------------
queen - king + prince  →  expected: princess
Result:  princess     0.6948
----------------------------------------
actress - actor + waiter  →  expected: waitress
Result:  waitress     0.7234
----------------------------------------
teacher - teach + drive  →  expected: driver
Result:  drives       0.5640
----------------------------------------
happier - happy + cold  →  expected: colder
Result:  colder       0.6168
----------------------------------------
strongest - strong + weak  →  expected: weakest
Result:  weakest      0.6113
----------------------------------------
writing - write + swim  →  expected: swimming
Result:  swimming     0.6603
-


# Section 2: Geometry of Fine-Tuning — Measuring ΔM and ΔD During Training

## Context
In recent papers on **parameter-efficient fine-tuning (PEFT)**, authors analyze *how weights move* during training. Two common geometry-inspired quantities are:

- **ΔM (magnitude change)**: how much *column norms* change.
- **ΔD (direction change)**: how much *column directions* change, often via cosine similarity.

In this homework, you will replicate a lightweight version of a "paper-style" analysis on a small BERT model, comparing:
1. **Full fine-tuning**
2. **LoRA(https://arxiv.org/abs/2106.09685)**
3. **DoRA(https://arxiv.org/abs/2402.09353)**

You will:
- fine-tune on **AG News (4-class)**,
- capture intermediate weight snapshots for **attention query/value** projection matrices,
- compute ΔM and ΔD over time,
- visualize the relationship between ΔM and ΔD,
- and answer analysis questions about what you observe.

## Learning Objectives
By the end, you should be able to:
- Implement a reproducible NLP training pipeline with `datasets` and `transformers`.
- Implement *column-wise* geometry metrics for weight updates.
- Instrument training to capture intermediate model states.
- Compare training-time geometry across optimization regimes (Full FT vs PEFT).
- Communicate empirical findings with appropriate plots and statistical summaries.

## What you must submit
1. This notebook with all **@ToDo** items completed.
2. All plots requested in the notebook.
3. Short written answers to the **analysis questions** at the end.

## Rules
- Do not change the definitions of ΔM and ΔD beyond what is specified (unless explicitly asked).
- You may tune hyperparameters *only* where the notebook indicates it is allowed.
- Your analysis must be based on your results (not copied from others).


---
## Step 0 — Environment and Setup

### What you should do and why
You will install the required libraries and set deterministic seeds. Reproducibility is essential for comparing geometry across methods: if runs are not controlled, you may attribute differences to LoRA/DoRA when they are actually caused by randomness.


In [ ]:

# ============================================================
# LIGHT setup that works with datasets==4.0.0:
# Model: google/bert_uncased_L-4_H-256_A-4  (Tiny BERT ~14M params)
# Dataset: ag_news (4-class classification; no dataset script)
# Methods: Full FT vs LoRA vs DoRA (if supported)
# Paper-style geometry: ΔM / ΔD on attention QUERY/VALUE weights, column-wise
# Compute points across intermediate steps (like paper) and plot:
#   - Scatter ΔD vs ΔM (per method)
#   - Density-like (hexbin) optional
#   - Correlation table (Pearson/Spearman)
# ============================================================

# If needed (uncomment):
# !pip -q install transformers datasets evaluate accelerate peft

import os, re, time, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import evaluate
import peft
from peft import LoraConfig, get_peft_model

# -------------------------
# 0.1) Reproducibility / device
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

import transformers, datasets
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)



---
## Step 1 — Load the Dataset (AG News)

### What you should do and why
You will load **AG News** and create smaller train/eval splits to keep the homework runnable on modest hardware. The goal is not to maximize accuracy, but to **compare geometry across training methods under similar conditions**.


In [ ]:

# -------------------------
# 1) Dataset: AG News (script-free)
# -------------------------
ds = load_dataset("ag_news")

TRAIN_SIZE = 8000 if device == "cuda" else 1500
EVAL_SIZE  = 1500 if device == "cuda" else 400

TRAIN_SIZE = min(TRAIN_SIZE, len(ds["train"]))
EVAL_SIZE  = min(EVAL_SIZE, len(ds["test"]))

train_ds = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
eval_ds  = ds["test"].shuffle(seed=SEED).select(range(EVAL_SIZE))
print("train size:", len(train_ds), "eval size:", len(eval_ds))

# Inspect a few examples (sanity check)
for i in range(2):
    print(f"\nExample {i}:")
    print("label:", train_ds[i]["label"])
    print("text :", train_ds[i]["text"][:200], "…")



---
## Step 2 — Tokenization and Dataset Preprocessing

### What you should do and why
Transformers expect tokenized inputs. You will implement a batching tokenizer function that:
- tokenizes text with truncation,
- sets a maximum length,
- and creates a `labels` field required by `Trainer`.

This step is foundational: incorrect labels or tokenization will invalidate the training and the geometry analysis.


In [ ]:

# -------------------------
# 2) Model + Tokenizer
# -------------------------
model_name = "google/bert_uncased_L-4_H-256_A-4"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_with_labels(batch):
    """
    Tokenize a batch of examples and attach labels.
    """
    
    # Tokenize the text
    output = tokenizer(batch["text"], truncation=True, max_length=192)
    
    # The Trainer looks for a column named "labels" to compute loss automatically
    output["labels"] = batch["label"]
    
    return output

# Apply tokenization
train_tok = train_ds.map(tokenize_with_labels, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize_with_labels,  batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok[0].keys())



---
## Step 3 — Define Evaluation Metrics

### What you should do and why
Accuracy is a simple sanity metric for AG News. You will implement the `compute_metrics` callback that converts logits into predicted class IDs.

This is used to:
- compare Full FT vs LoRA vs DoRA in terms of *task performance*,
- and ensure training is actually working before interpreting geometry plots.


In [ ]:

# -------------------------
# 3) Metrics
# -------------------------
acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """
    eval_pred is (logits, labels).
    Return a dict like {"accuracy": value}.
    """
    logits, labels = eval_pred
    
    # Logits are the raw scores from the model. 
    # We take the index of the highest score as our predicted class.
    predictions = np.argmax(logits, axis=-1)
    
    # Returns a dictionary: {"accuracy": 0.XX}
    return acc.compute(predictions=predictions, references=labels)



---
## Step 4 — Training Helpers (Parameters, Arguments)

### What you should do and why
You will implement lightweight helper functions:
- count parameters (total vs trainable),
- create TrainingArguments compatible with older/newer Transformers.

These utilities help quantify the *efficiency* of PEFT methods and keep the training loop clean.


In [ ]:

# -------------------------
# 4) Training helpers
# -------------------------
def count_params(model):
    """
    Return (total_params, trainable_params).
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    return total_params, trainable_params
    

def short_model_report(model, title="model"):
    total, trainable = count_params(model)
    print(f"[{title}] total params: {total:,}")
    print(f"[{title}] trainable params: {trainable:,} ({trainable/total:.4%})")

def make_training_args(outdir, lr, epochs, batch=32, eval_every="epoch"):
    """
    Create TrainingArguments with backward-compatible naming.

    Note: transformers changed `evaluation_strategy` -> `eval_strategy` in some versions.
    """
    base = dict(
        output_dir=outdir,
        report_to=[],
        logging_steps=50,
        save_strategy="no",
        fp16=(device=="cuda"),
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="cosine",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch if device=="cuda" else max(8, batch//2),
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=1,
    )
    try:
        return TrainingArguments(evaluation_strategy=eval_every, **base)
    except TypeError:
        return TrainingArguments(eval_strategy=eval_every, **base)



---
## Step 5 — Implement Paper-Style ΔM and ΔD (Column-Wise)

### What you should do and why
You will implement the *core geometry metrics*:

- Let W ∈ R^{d_out × d_in}.
- Consider each **column** as a vector in R^{d_out}.

Definitions (column-wise):
- Column norms: ||W[:, j]||_2
- **ΔM**: mean absolute change in column norms from initialization
- **ΔD**: 1 − mean cosine similarity between current and initial columns

These are used in the final scatter plot of ΔD vs ΔM and should match the definitions in the provided design.


In [ ]:

# -------------------------
# 5) Paper-style ΔM / ΔD definitions (column-wise)
# -------------------------
def col_norms(W: torch.Tensor) -> torch.Tensor:
    """
    Return a vector of L2 norms for each column of W.
    """
    return torch.linalg.vector_norm(W.float(), ord=2, dim=0)


def mean_cosine_cols(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """
    Compute mean cosine similarity between corresponding columns of A and B.
    """
    # Normalize columns
    eps = 1e-8
    norm_A = col_norms(A).clamp(min=eps)
    norm_B = col_norms(B).clamp(min=eps)
    
    A_n = A / norm_A
    B_n = B / norm_B
    
    cosines = (A_n * B_n).sum(dim=0) # Cosine similarity for each column is the dot product of normalized columns
    
    return cosines.mean()


def delta_M_D(W0: torch.Tensor, Wt: torch.Tensor):
    """
    Compute (ΔM, ΔD) between initial weight W0 and current weight Wt.

    Definitions:
      m0 = column norms of W0
      mt = column norms of Wt
      ΔM = mean |mt - m0|
      cos_mean = mean cosine similarity between columns of Wt and W0
      ΔD = 1 - cos_mean
    """
    m0 = col_norms(W0)
    mt = col_norms(Wt)
    
    # ΔM = mean absolute change in column norms
    delta_m = torch.abs(mt - m0).mean().item()
    
    # ΔD = 1 - mean cosine similarity
    cos_mean = mean_cosine_cols(W0, Wt).item()
    delta_d = 1 - cos_mean
    
    return delta_m, delta_d



---
## Step 6 — Capture Intermediate Weight Snapshots (Query/Value Only)

### What you should do and why
We need intermediate snapshots across training (not only at the end). You will:

1. Identify **target parameter names** matching BERT attention query/value weights:
   - `bert.encoder.layer.{i}.attention.self.query.weight`
   - `bert.encoder.layer.{i}.attention.self.value.weight`

2. Implement a TrainerCallback that captures these tensors at selected training steps.

This instrumentation enables time-resolved geometry analysis similar to paper plots showing how weights evolve *during* fine-tuning.


In [ ]:

# -------------------------
# 6) Capture intermediate snapshots (only query/value weights)
# -------------------------
TARGET_REGEX = re.compile(r"bert\.encoder\.layer\.\d+\.attention\.self\.(query|value)\.weight$")

def select_target_param_names(model):
    """
    Return sorted parameter names that match TARGET_REGEX.
    """
    target_names = []
    for name, _ in model.named_parameters():
        if TARGET_REGEX.search(name):
            target_names.append(name)
    return sorted(target_names)


class WeightSnapshotCallback(TrainerCallback):
    """Captures a CPU clone of selected parameters at specified training steps."""
    def __init__(self, param_names, steps_to_capture):
        self.param_names = set(param_names)
        self.steps_to_capture = set(steps_to_capture)
        self.snapshots = {}  # step -> {name: tensor_cpu}

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step in self.steps_to_capture and step not in self.snapshots:
            model = kwargs['model']
            step_snap = {}
            for name, p in model.named_parameters():
                if name in self.param_names:
                    step_snap[name] = p.detach().cpu().clone()
            self.snapshots[step] = step_snap

def choose_capture_steps(total_steps):
    """
    Choose capture steps similar to paper-style intermediate points.
    """
    percentages = [0.0, 0.2, 0.5, 0.8, 1.0]
    steps = sorted(list(set(int(p * total_steps) for p in percentages)))
    return steps


---
## Step 7 — Training With Snapshots

### What you should do and why
You will write a training wrapper that:
- creates a callback with selected capture steps,
- stores an explicit step-0 snapshot (initial weights),
- trains with `Trainer.train()`,
- evaluates at the end,
- and returns snapshots + metrics.

This wrapper ensures the only difference between methods is **which parameters are trained**, not how we collect geometry.


In [ ]:

# -------------------------
# 7) Training loop with snapshots
# -------------------------
def train_with_snapshots(model, outdir, lr, epochs, batch=32):
    """
    Train a model while capturing intermediate snapshots of query/value weights.
    """
    model.to(device)
    
    # 1. Identify which weights we are watching
    target_names = select_target_param_names(model)
    
    # 2. Calculate steps to determine when to take "photos"
    steps_per_epoch = math.ceil(len(train_tok) / (batch if device=="cuda" else max(8, batch//2)))
    total_steps = steps_per_epoch * epochs
    capture_steps = choose_capture_steps(total_steps)
    
    # 3. Setup Callback
    cb = WeightSnapshotCallback(target_names, capture_steps)
    
    # 4. Manual Step-0 Snapshot (Initial state)
    step_0_snap = {}
    for name, p in model.named_parameters():
        if name in target_names:
            step_0_snap[name] = p.detach().cpu().clone()
    cb.snapshots[0] = step_0_snap
    
    # 5. Create Trainer
    args = make_training_args(outdir, lr, epochs, batch=batch)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[cb]
    )
    
    # 6. Train and Evaluate
    start_time = time.time()
    train_result = trainer.train()
    total_time = time.time() - start_time
    
    metrics = trainer.evaluate()
    metrics["train_runtime_seconds"] = total_time
    
    return trainer, metrics, cb.snapshots



---
## Step 8 — Build ΔM/ΔD Point Cloud

### What you should do and why
After training, you will convert snapshots into a table where each row corresponds to:
- a method (full/lora/dora),
- a training step,
- a specific weight matrix name,
- and its geometry relative to initialization.

This tabular representation supports scatter plots, density plots, and correlation analysis.


In [ ]:

# -------------------------
# 8) Build ΔM/ΔD points
# -------------------------
def build_points(snapshots, method):
    """
    Create a DataFrame with columns:
      method, step, name, dM, dD

    For each captured step and each target weight matrix:
      compute (ΔM, ΔD) between W0 and Wt.
    """
    rows = []
    sorted_steps = sorted(snapshots.keys())
    
    # Step 0 is our reference point (initial weights)
    W0_dict = snapshots[0]
    
    for step in sorted_steps:
        # We usually skip step 0 for plotting since delta is always 0
        if step == 0: continue
        
        current_snap = snapshots[step]
        for name in W0_dict.keys():
            if name in current_snap:
                # Use our previously defined geometric function
                dm, dd = delta_M_D(W0_dict[name], current_snap[name])
                
                rows.append({
                    "method": method,
                    "step": step,
                    "name": name,
                    "dM": dm,
                    "dD": dd
                })
                
    return pd.DataFrame(rows)


---
## Step 9 — Correlation Utilities (Pearson and Spearman)

### What you should do and why
A typical paper-style question is: *are magnitude and direction changes related?*

You will implement:
- Pearson correlation (linear association),
- Spearman correlation (rank-based, monotonic association).

These will be computed per method on the point cloud.


In [ ]:

# -------------------------
# 9) Correlation helpers
# -------------------------
def pearson(x, y):
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    
    # Filter out NaNs or infinite values for stability
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    
    if len(x) < 3:
        return np.nan
    
    return np.corrcoef(x, y)[0, 1]

def spearman(x, y):
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    
    if len(x) < 3:
        return np.nan
    
    def get_ranks(v):
        return pd.Series(v).rank(method='average').values

    rank_x = get_ranks(x)
    rank_y = get_ranks(y)
    
    # Spearman is just the Pearson correlation of the ranks
    return pearson(rank_x, rank_y)



---
## Step 10 — Define Methods: Full FT vs LoRA vs DoRA

### What you should do and why
You will instantiate and train three variants:

1. **Full fine-tuning**: all parameters trainable.
2. **LoRA**: train low-rank adapters on attention query/value modules.
3. **DoRA**: if supported, uses a decomposition that can alter geometry differently than LoRA.

You will record:
- number of trainable parameters,
- final eval accuracy,
- training time,
- and geometry point clouds.


In [ ]:

# -------------------------
# 10) Hyperparameters (adjust ONLY if runtime requires)
# -------------------------
EPOCHS_FULL = 3
EPOCHS_ADAPTER = 6
LR_FULL = 2e-5
LR_ADAPTER = 3e-4
BATCH = 32

def make_lora_model(model_name, num_labels=4, use_dora=False):
    """
    Create a (LoRA or DoRA) model for sequence classification.
    """
    # Load base model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    # Create LoraConfig
    config = LoraConfig(
        r=8, 
        lora_alpha=16, 
        lora_dropout=0.05,
        bias="none", 
        task_type="SEQ_CLS",
        target_modules=["query", "value"],
        use_dora=use_dora  # This is the key switch between standard LoRA and DoRA
    )
    
    # Wrap with PEFT
    peft_model = get_peft_model(model, config)
    return peft_model

In [ ]:

# -------------------------
# 10.1) Train: Full FT
# -------------------------
full_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
short_model_report(full_model, "full-ft (before)")

trainer_full, metrics_full, snaps_full = train_with_snapshots(
    full_model,
    outdir="out_full_agnews_bert4",
    lr=LR_FULL,
    epochs=EPOCHS_FULL,
    batch=BATCH
)
print("full metrics:", metrics_full)
print("full captured steps:", sorted(snaps_full.keys()))


In [ ]:

# -------------------------
# 10.2) Train: LoRA
# -------------------------
lora_model = make_lora_model(model_name, num_labels=4, use_dora=False)
short_model_report(lora_model, "lora (before)")
try:
    lora_model.print_trainable_parameters()
except Exception as e:
    print("Could not print trainable parameters:", e)

trainer_lora, metrics_lora, snaps_lora = train_with_snapshots(
    lora_model,
    outdir="out_lora_agnews_bert4",
    lr=LR_ADAPTER,
    epochs=EPOCHS_ADAPTER,
    batch=BATCH
)
print("lora metrics:", metrics_lora)
print("lora captured steps:", sorted(snaps_lora.keys()))


In [ ]:

# -------------------------
# 10.3) Train: DoRA (if supported)
# -------------------------
snaps_dora = None
metrics_dora = {"skipped": True}

try:
    dora_model = make_lora_model(model_name, num_labels=4, use_dora=True)
    short_model_report(dora_model, "dora (before)")
    try:
        dora_model.print_trainable_parameters()
    except Exception as e:
        print("Could not print trainable parameters:", e)

    trainer_dora, metrics_dora, snaps_dora = train_with_snapshots(
        dora_model,
        outdir="out_dora_agnews_bert4",
        lr=LR_ADAPTER,
        epochs=EPOCHS_ADAPTER,
        batch=BATCH
    )
    print("dora metrics:", metrics_dora)
    print("dora captured steps:", sorted(snaps_dora.keys()))

except TypeError as e:
    print("DoRA not supported by this PEFT version. Skipping DoRA.")
    print("Error:", e)



---
## Step 11 — Assemble All Points and Compute Correlations

### What you should do and why
You will build a combined DataFrame for all methods and focus on **query weights** (often closest to paper plots).

Then you will compute correlation summaries:
- Pearson(ΔD, ΔM)
- Spearman(ΔD, ΔM)

These summaries help quantify trends you see in scatter plots.


In [ ]:

# -------------------------
# 11) Build ΔM/ΔD points + correlations
# -------------------------
df_full = build_points(snaps_full, "full")
df_lora = build_points(snaps_lora, "lora")

dfs = [df_full, df_lora]
if snaps_dora is not None:
    df_dora = build_points(snaps_dora, "dora")
    dfs.append(df_dora)

df = pd.concat(dfs, ignore_index=True)
print("points:", len(df))

# Focus on QUERY only (often matches paper plots better)
df_q = df[df["name"].str.contains(r"\.query\.weight$", regex=True)].copy()
print("points (query only):", len(df_q))

summary = []
for m in sorted(df_q["method"].unique()):
    sub = df_q[df_q["method"] == m]
    summary.append({
        "method": m,
        "n_points": len(sub),
        "pearson(dD, dM)": pearson(sub["dD"], sub["dM"]),
        "spearman(dD, dM)": spearman(sub["dD"], sub["dM"]),
    })
summary_df = pd.DataFrame(summary)
summary_df



---
## Step 12 — Plots: Scatter and Hexbin (Paper-Style)

### What you should do and why
You will plot ΔD vs ΔM for each method.
- Scatter plots show raw point clouds.
- Hexbin plots provide a density-like view similar to many papers.

Include Pearson and Spearman in the plot titles to connect visual patterns with quantitative summaries.


In [ ]:

def scatter_all(df_use, title_suffix="(query weights)"):
    """
    Creates scatter plots for each method to show the distribution of weight updates.
    """
    methods = df_use['method'].unique()
    fig, axes = plt.subplots(1, len(methods), figsize=(6 * len(methods), 5))
    if len(methods) == 1: axes = [axes]
    
    for ax, method in zip(axes, methods):
        sub = df_use[df_use['method'] == method]
        x = sub['dD'].values
        y = sub['dM'].values
        
        # Calculate Correlations
        r = pearson(x, y)
        rs = spearman(x, y)
        
        ax.scatter(x, y, alpha=0.4, s=10, edgecolors='none')
        ax.set_title(f"{method} {title_suffix}\nPearson: {r:.3f} | Spearman: {rs:.3f}")
        ax.set_xlabel("Direction Change (ΔD)")
        ax.set_ylabel("Magnitude Change (ΔM)")
        ax.grid(True, linestyle='--', alpha=0.6)
        
    plt.tight_layout()
    plt.show()

def hexbin_all(df_use, gridsize=35, title_suffix="(query weights)"):
    """
    Creates hexbin density plots to highlight where the majority of updates cluster.
    """
    methods = df_use['method'].unique()
    fig, axes = plt.subplots(1, len(methods), figsize=(6 * len(methods), 5))
    if len(methods) == 1: axes = [axes]
    
    for ax, method in zip(axes, methods):
        sub = df_use[df_use['method'] == method]
        x = sub['dD'].values
        y = sub['dM'].values
        
        r = pearson(x, y)
        rs = spearman(x, y)
        
        # mincnt=1 ensures we don't color background hexagons with 0 points
        hb = ax.hexbin(x, y, gridsize=gridsize, cmap='viridis', mincnt=1)
        fig.colorbar(hb, ax=ax, label='Count')
        
        ax.set_title(f"{method} {title_suffix}\nPearson: {r:.3f} | Spearman: {rs:.3f}")
        ax.set_xlabel("Direction Change (ΔD)")
        ax.set_ylabel("Magnitude Change (ΔM)")
        
    plt.tight_layout()
    plt.show()
    


In [ ]:

# Produce plots (once you implement plotting functions)
scatter_all(df_q)
hexbin_all(df_q)



---
## Step 13 — Training Metrics Summary Table

### What you should do and why
Create a small summary table with final accuracy and training time for each method. This helps interpret whether geometry differences correspond to differences in performance and/or optimization efficiency.


In [ ]:

train_results = [
    {"method": "full", "eval_accuracy": metrics_full.get("eval_accuracy"), "train_seconds": metrics_full.get("train_seconds")},
    {"method": "lora", "eval_accuracy": metrics_lora.get("eval_accuracy"), "train_seconds": metrics_lora.get("train_seconds")},
]
if snaps_dora is None:
    train_results.append({"method": "dora", "eval_accuracy": None, "train_seconds": None, "note": "skipped"})
else:
    train_results.append({"method": "dora", "eval_accuracy": metrics_dora.get("eval_accuracy"), "train_seconds": metrics_dora.get("train_seconds")})

pd.DataFrame(train_results)



---
# Analysis Questions (Answer in Markdown)

Answer concisely but precisely. Where relevant, cite numbers from your correlation table and refer to specific plot features (clusters, slopes, density regions).

1. **Sanity checks**
   1. What accuracy did you obtain for each method?
   2. Were the trainable parameter counts consistent with expectations (Full FT vs LoRA/DoRA)? Provide the counts.

2. **Geometry–performance relationship**
   1. Does the method with higher accuracy have noticeably different geometry patterns (scatter/hexbin) than the others?
   2. Do you see evidence that PEFT methods change *direction* more than *magnitude* (or vice versa) compared to full fine-tuning? Explain.

3. **Correlation interpretation**
   1. Compare Pearson vs Spearman for each method. Are they similar? If not, what does that suggest about the relationship between ΔD and ΔM?
   2. Which method has the strongest association between ΔD and ΔM? Provide correlation values.

4. **Step-wise dynamics**
   1. Look at points grouped by step (you may color points by step if you want). Do early steps differ from late steps?
   2. Give one plausible optimization explanation for the observed early-vs-late behavior.

5. **Query vs Value**
   1. Repeat the analysis for **value weights** only (`.value.weight`). Do conclusions change?
   2. Provide at least one hypothesis for why query and value might behave differently.

6. **Method critique**
   1. Identify at least two limitations of this experiment design (e.g., dataset size, step sampling, metric choices).
   2. Propose one extension that would strengthen the evidence (e.g., multiple seeds, additional layers, other target modules).

7. **Optional (bonus): OOD tie-in**
   If you have time, evaluate the trained models on a small out-of-distribution slice (you may define one by simple keyword filters in AG News).
   1. Do geometry patterns correlate with OOD robustness in your quick test?
   2. What additional experiments would be needed to make a credible claim?
